In [7]:
import pandas as pd
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout, LSTM
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

In [10]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
#loading data
DATA_PATH  = "../ids/dataset.csv"          
LABEL_COL  = "label"                     
DROP_COLS  = ["ip_source", "ip_dest", "ts"]   # leakage columns
TEST_SIZE  = 0.20
BATCH_SIZE = 256
EPOCHS     = 30
PATIENCE   = 5

In [ ]:

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
 
X = df.drop(columns=[LABEL_COL]).values
y_raw = df[LABEL_COL].values
 
le = LabelEncoder()
y = le.fit_transform(y_raw)
n_classes = len(le.classes_)
 
print(f"Samples : {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes : {n_classes}")

Samples : 39778
Features: 46
Classes : 2


In [ ]:
#split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED,
)

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)
 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
 
n_features = X_train.shape[1]
 
y_train_oh = to_categorical(y_train, num_classes=n_classes)
y_test_oh  = to_categorical(y_test,  num_classes=n_classes)

In [ ]:
# reshape per model family
X_train_3d = X_train.reshape(-1, n_features, 1)
X_test_3d  = X_test.reshape(-1, n_features, 1)

In [17]:
# ConvLSTM needs 5D: (samples, time, rows, cols, channels).
# Reshape features into T blocks of (R x C); pad with zeros if needed.
T = 4
block       = int(np.ceil(n_features / T))
R           = int(np.ceil(np.sqrt(block)))
C           = int(np.ceil(block / R))
pad_needed  = T * R * C - n_features
 
X_train_pad = np.pad(X_train, ((0, 0), (0, pad_needed)))
X_test_pad  = np.pad(X_test,  ((0, 0), (0, pad_needed)))
X_train_5d  = X_train_pad.reshape(-1, T, R, C, 1)
X_test_5d   = X_test_pad.reshape(-1, T, R, C, 1)
 
print(f"3D shape: {X_train_3d.shape}   5D shape: {X_train_5d.shape}")

3D shape: (31822, 46, 1)   5D shape: (31822, 4, 4, 3, 1)


In [ ]:
#Build LSTM
def build_lstm():
    m = Sequential([
        Input(shape=(n_features, 1)),
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dense(n_classes, activation="softmax"),
    ], name="LSTM")
    m.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
    return m

In [ ]:
#Build CNN
def build_cnn():
    m = Sequential([
        Input(shape=(n_features, 1)),
        Conv1D(64,  kernel_size=3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Conv1D(128, kernel_size=3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Flatten(),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(n_classes, activation="softmax"),
    ], name="CNN")
    m.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
    return m

In [ ]:
#Build CNN-LSTM
def build_cnn_lstm():
    m = Sequential([
        Input(shape=(n_features, 1)),
        Conv1D(64,  kernel_size=3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Conv1D(128, kernel_size=3, activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dense(n_classes, activation="softmax"),
    ], name="CNN-LSTM")
    m.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
    return m

In [ ]:
#Build CONV-LSTM
def build_convlstm():
    m = Sequential([
        Input(shape=(T, R, C, 1)),
        ConvLSTM2D(filters=32, kernel_size=(2, 2), padding="same",
                   activation="tanh", return_sequences=True),
        BatchNormalization(),
        ConvLSTM2D(filters=64, kernel_size=(2, 2), padding="same",
                   activation="tanh", return_sequences=False),
        BatchNormalization(),
        Flatten(),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(n_classes, activation="softmax"),
    ], name="ConvLSTM")
    m.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
    return m

In [ ]:
#Built evaluation function
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    es = EarlyStopping(monitor="val_loss",
                       patience=PATIENCE,
                       restore_best_weights=True)
    print(f"\n Training {name}")
    model.fit(X_tr, y_tr,
              validation_split=0.10,
              epochs=EPOCHS,
              batch_size=BATCH_SIZE,
              callbacks=[es],
              verbose=2)
 
    y_pred = np.argmax(model.predict(X_te, batch_size=BATCH_SIZE, verbose=0),
                       axis=1)
    y_true = np.argmax(y_te, axis=1)
 
    return {
        "Model":     name,
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall":    recall_score(   y_true, y_pred, average="macro", zero_division=0),
        "F1":        f1_score(       y_true, y_pred, average="macro", zero_division=0),
    }

In [26]:
results = [
    evaluate("LSTM",     build_lstm(),     X_train_3d, y_train_oh, X_test_3d, y_test_oh),
    evaluate("CNN",      build_cnn(),      X_train_3d, y_train_oh, X_test_3d, y_test_oh),
    evaluate("CNN-LSTM", build_cnn_lstm(), X_train_3d, y_train_oh, X_test_3d, y_test_oh),
    evaluate("ConvLSTM", build_convlstm(), X_train_5d, y_train_oh, X_test_5d, y_test_oh),
]


 Training LSTM
Epoch 1/30
112/112 - 17s - 148ms/step - accuracy: 0.9484 - loss: 0.2153 - val_accuracy: 0.9529 - val_loss: 0.1232
Epoch 2/30
112/112 - 15s - 133ms/step - accuracy: 0.9575 - loss: 0.1096 - val_accuracy: 0.9617 - val_loss: 0.0919
Epoch 3/30
112/112 - 13s - 118ms/step - accuracy: 0.9617 - loss: 0.0949 - val_accuracy: 0.9604 - val_loss: 0.0923
Epoch 4/30
112/112 - 13s - 115ms/step - accuracy: 0.9621 - loss: 0.0900 - val_accuracy: 0.9629 - val_loss: 0.0825
Epoch 5/30
112/112 - 12s - 109ms/step - accuracy: 0.9623 - loss: 0.0851 - val_accuracy: 0.9636 - val_loss: 0.0824
Epoch 6/30
112/112 - 12s - 110ms/step - accuracy: 0.9620 - loss: 0.0820 - val_accuracy: 0.9632 - val_loss: 0.0791
Epoch 7/30
112/112 - 12s - 109ms/step - accuracy: 0.9626 - loss: 0.0801 - val_accuracy: 0.9623 - val_loss: 0.0781
Epoch 8/30
112/112 - 12s - 109ms/step - accuracy: 0.9624 - loss: 0.0819 - val_accuracy: 0.9617 - val_loss: 0.0775
Epoch 9/30
112/112 - 15s - 135ms/step - accuracy: 0.9623 - loss: 0.0761 

In [27]:
df_results = pd.DataFrame(results)
metric_cols = ["Accuracy", "Precision", "Recall", "F1"]
df_results[metric_cols] = df_results[metric_cols].round(4)
 
print("\n FINAL COMPARISON")
print(df_results.to_string(index=False))


 FINAL COMPARISON
   Model  Accuracy  Precision  Recall     F1
    LSTM    0.9595     0.8268  0.8329 0.8298
     CNN    0.9696     0.8646  0.8849 0.8744
CNN-LSTM    0.9715     0.8609  0.9213 0.8883
ConvLSTM    0.9679     0.8550  0.8859 0.8696
